In [3]:
%%html
<video controls autoplay><source src="https://huggingface.co/sb3/ppo-LunarLander-v2/resolve/main/replay.mp4" type="video/mp4"></video>

1. **Gymnasium** is a Python library that provides standardized environments for training and testing reinforcement learning agents. Instead of creating a simulation yourself, Gymnasium provides ready-made environments with a consistent interface.

2. **Stable-Baselines3** is a Python library that contains implementations of popular reinforcement learning algorithms. Instead of coding algorithms like PPO or DQN from scratch, you simply import them.

3. **LunarLander-v2** is one of Gymnasium's classic reinforcement learning environments. The goal is to land a spacecraft safely on a landing pad.

In [ ]:
! pip install gymnasium[box2d]
! pip install stable-baselines3[extra]
! pip install huggingface_sb3
!sudo apt-get update
!sudo apt-get install -y python3-opengl
!apt install ffmpeg
!apt install xvfb
!pip3 install pyvirtualdisplay

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [2]:
import gymnasium
from stable_baselines3 import PPO # Proximal Policy Optimization.
from stable_baselines3.common.env_util import make_vec_env # It creates multiple environments running in parallel.
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [4]:
import gymnasium as gym

env = gym.make("LunarLander-v3")

observation , info = env.reset()

for _ in range(20) :
  action = env.action_space.sample() # random action

  print("Action taken:", action)

  observation,reward,terminated,truncated,info = env.step( action)

  if terminated or truncated :
    print("Environment is reset")
    observation, info = env.reset()
env.close()



Action taken: 2
Action taken: 3
Action taken: 0
Action taken: 0
Action taken: 0
Action taken: 0
Action taken: 2
Action taken: 2
Action taken: 2
Action taken: 3
Action taken: 3
Action taken: 2
Action taken: 2
Action taken: 3
Action taken: 2
Action taken: 3
Action taken: 0
Action taken: 0
Action taken: 1
Action taken: 3


<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


There are four discrete actions available:
- 0: do nothing
- 1: fire left orientation engine
- 2: fire main engine
- 3: fire right orientation engine

The state is an 8-dimensional vector: the coordinates of the lander in x & y, its linear velocities in x & y, its angle, its angular velocity, and two booleans that represent whether each leg is in contact with the ground or not.

In [5]:
env = gym.make("LunarLander-v3")
env.reset()

print("_____OBSERVATION SPACE_____ \n")
print("Observation Space Shape", env.observation_space.shape)
print("Sample observation", env.observation_space.sample())

_____OBSERVATION SPACE_____ 

Observation Space Shape (8,)
Sample observation [-0.6168007  -0.970424    5.2447786   9.609208   -1.1081978   6.3592706
  0.13437746  0.46155035]


In [6]:
print("\n _____ACTION SPACE_____ \n")
print("Action Space Shape", env.action_space.n)
print("Action Space Sample", env.action_space.sample())


 _____ACTION SPACE_____ 

Action Space Shape 4
Action Space Sample 1


---

A **step** is one single interaction between the agent and the environment.

An **episode** is a complete run from the beginning of the environment until a terminal condition occurs.

In [8]:
env = make_vec_env('LunarLander-v3', n_envs=16) # stacking multiple independent environments into a single environment

---

MLP = Multi-Layer Perceptron

In PPO, MlpPolicy actually contains two neural networks (actor-critic architecture):

                 Observation
                     |
                     v
              Feature extractor
                     |
          +----------+----------+
          |                     |
          v                     v
       Actor                  Critic
   (Policy network)       (Value network)
          |                     |
          v                     v
   Action probabilities     State value



In [9]:
env = gym.make('LunarLander-v3')

model = PPO (
    policy='MlpPolicy',
    env=env,
    n_steps=1024,
    batch_size = 64,
    n_epochs=4,
    gamma=0.999,
    gae_lambda=0.98,
    ent_coef=0.01,
    verbose=1
)

model.learn(total_timesteps=1000000)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Streaming output truncated to the last 5000 lines.
|    value_loss           | 26.9         |
------------------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 397           |
|    ep_rew_mean          | 221           |
| time/                   |               |
|    fps                  | 558           |
|    iterations           | 740           |
|    time_elapsed         | 1357          |
|    total_timesteps      | 757760        |
| train/                  |               |
|    approx_kl            | 0.00010571745 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -0.632        |
|    explained_variance   | 0.898         |
|    learning_rate        | 0.0003        |
|    loss                 | 19.1          |
|    n_updates            | 2956          |
|    policy_gradient_loss | 2.83e-05      |
|    value_loss           |

In [10]:
eval_env = Monitor(gym.make("LunarLander-v3", render_mode='rgb_array'))

mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)

print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")

mean_reward=180.51 +/- 98.00392852122181
